In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.6 MB/s eta 0:00:00


In [ ]:
# Load Libraries
import cv2
import numpy as np
import joblib
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# Load models
pose_model = YOLO("/content/best (1).pt")
clf = joblib.load("/content/fall_classifier_rf.pkl")

# Load video
cap = cv2.VideoCapture("/content/fall_video.mp4")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model.predict(source=frame, conf=0.4, verbose=False)

    for r in results:

        if r.boxes is None or r.keypoints is None:
            continue

        boxes = r.boxes.xyxy.cpu().numpy()
        keypoints = r.keypoints.xy.cpu().numpy()

        num_people = min(len(boxes), len(keypoints))

        for i in range(num_people):

            kp = keypoints[i]
            x1, y1, x2, y2 = map(int, boxes[i])

            label_text = "Unknown"
            color = (0, 255, 0)

            required_ids = [5, 6, 11, 12]

            # Check valid keypoints
            if all(kp[j][0] != 0 and kp[j][1] != 0 for j in required_ids):

                # Feature Extraction
                shoulder_mid = (kp[5] + kp[6]) / 2
                hip_mid = (kp[11] + kp[12]) / 2

                dx = hip_mid[0] - shoulder_mid[0]
                dy = hip_mid[1] - shoulder_mid[1]

                angle = abs(np.degrees(np.arctan2(dy, dx)))
                if angle > 90:
                    angle = 180 - angle

                body_h = kp[:,1].max() - kp[:,1].min()
                body_w = kp[:,0].max() - kp[:,0].min()
                ratio = body_w / (body_h + 1e-6)

                center_y = np.mean(kp[:,1])
                image_height = frame.shape[0]
                center_norm = center_y / image_height

                shoulder_slope = abs(kp[5][1] - kp[6][1])
                hip_slope = abs(kp[11][1] - kp[12][1])

                features = np.array([[angle, ratio, center_norm, shoulder_slope, hip_slope]])

                # Prediction
                pred = clf.predict(features)[0]

                if pred == 0:
                    label_text = "FALL DETECTED"
                    color = (0, 0, 255)
                else:
                    label_text = "Not Fallen"
                    color = (0, 255, 0)

            else:
                label_text = "Keypoints Missing"
                color = (0, 255, 255)

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Draw label
            cv2.putText(frame, label_text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            # Draw keypoints
            for x, y in kp:
                if x != 0 and y != 0:
                    cv2.circle(frame, (int(x), int(y)), 3, (255, 0, 0), -1)

    cv2_imshow(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()